In [1]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

class DeepSeekFundAnalyst:
    def __init__(self):
        # 1. 配置 DeepSeek-Reasoner (R1) 模型

        self.llm = ChatOpenAI(
            model="deepseek-reasoner",
            openai_api_key="sk-84ec55da76944a109524df873d20d975",
            openai_api_base="https://api.deepseek.com",
        )

        # 2. 弹性阈值设定
        self.MAX_CHAR_LIMIT = 100000
        self.TOP_N_LEVEL1 = 20
        self.TOP_N_LEVEL2 = 10

        # 3. 提示词
        self.prompt_template = ChatPromptTemplate.from_messages([
            ("system", "你是一个资深的金融量化分析专家。你将获得一份基金的结构化数据报告，请根据数据进行深度审计与分析。"),
            ("user", """
            请阅读以下基金 {fund_code} 的数据报告，并严格按照以下四个维度进行分析。

            输出要求：总共不超过 250 字，语言简练专业。

            1. 资产配置趋势：分析债券、股票和现金的比例变化，评估财务稳定性和运营效率（重点指出是否存在大量或高频的仓位变动）。
            2. 风险分散评估：分析持股和债券持有的集中度与行业分布，评价其是否有效分散了系统性风险。
            3. 投资组合偏好：判断其属于“稳定增长型”（偏好债券/蓝筹）、“短期收入型”（偏好高回报/高波动股票）还是“平衡型”。
            4. 风格一致性审计：对比数据反映的实际持仓风格与基金声称的投资目标/策略是否一致。

            --- 基金数据报告 ---
            {report_content}
            """)
        ])

        self.chain = self.prompt_template | self.llm | StrOutputParser()

    def _elastic_compress(self, content):
        current_len = len(content)
        if current_len <= self.MAX_CHAR_LIMIT:
            return content

        print(f"⚠️ 数据长度 {current_len} 超过阈值，正在精简...")

        def filter_rows(text, max_seq):
            lines = text.split('\n')
            filtered = []
            for line in lines:
                if '|' in line:
                    parts = [p.strip() for p in line.split('|')]
                    try:
                        seq_val = next((int(p) for p in parts if p.isdigit()), None)
                        if seq_val and seq_val > max_seq:
                            continue
                    except:
                        pass
                filtered.append(line)
            return '\n'.join(filtered)

        content_v1 = filter_rows(content, self.TOP_N_LEVEL1)
        if len(content_v1) <= self.MAX_CHAR_LIMIT:
            return content_v1

        return filter_rows(content, self.TOP_N_LEVEL2)

    def analyze(self, fund_code, file_path):
        if not os.path.exists(file_path):
            return f"找不到文件: {file_path}"

        with open(file_path, "r", encoding="utf-8") as f:
            raw_data = f.read()

        final_input = self._elastic_compress(raw_data)

        try:
            #模型的最终结论.
            response = self.chain.invoke({
                "fund_code": fund_code,
                "report_content": final_input
            })
            return response
        except Exception as e:
            return f"DeepSeek-R1 调用失败: {str(e)}"

if __name__ == "__main__":
    analyst = DeepSeekFundAnalyst()
    result = analyst.analyze("000003", "Fund_Full_Report_000003.md")
    print("\n[DeepSeek-R1 深度分析结论]\n", result)

⚠️ 数据长度 247403 超过阈值，正在启动 R1 适配精简...

[DeepSeek-R1 深度分析结论]
 1.  **资产配置趋势**：债券仓位从2024年底的78.3%持续攀升至2025年底的92.18%，股票仓位则从最高7.9%（2024年9月）锐减至近零，现金比例波动剧烈（最高18.5%）。显示基金在报告期内进行了显著的降风险、去权益化操作，存在高频且大幅的仓位变动。
2.  **风险分散评估**：债券持仓集中度偏高，前五大债券占比常超30%；股票持仓（当存在时）行业集中于制造业与金融业，个股权重低。整体对单一债券风险暴露较高，但权益端风险因极低仓位而有限。
3.  **投资组合偏好**：属于“稳定增长型”。投资组合始终以债券为绝对核心（>78%），权益仓位极低且持续削减，完全符合债券型基金追求稳健收益的特征，无高波动偏好。
4.  **风格一致性审计**：基本一致，但存细微偏差。基金严格遵循了“债券型”及“主要投资可转换债券”的定位，债券仓位始终高于80%。偏差在于其股票投资并未完全遵循契约中“优先选择红利股票”的策略，持仓含普通蓝筹股。
